# NLP Experiment 5 – Topic modelling with LDA (20 Newsgroups)
**Runtime:** Python 3 (CPU). Runtime → Run all.

**LDA (Latent Dirichlet Allocation):** every document is a **mix of topics**, every topic is a **mix of words**.
LDA discovers the topics from word co-occurrence alone (unsupervised – no labels).
Dataset: 20 Newsgroups posts; we use 4 newsgroups so we can check whether LDA finds them.

In [ ]:
import re
import nltk
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

## Load documents

In [ ]:
categories = ["sci.space", "rec.sport.hockey", "talk.politics.guns", "comp.graphics"]
news = fetch_20newsgroups(subset="train", categories=categories,
                          remove=("headers", "footers", "quotes"), random_state=42)
documents = news.data
print("Documents:", len(documents))
print(documents[0][:400])

## (a) Preprocess: lowercase, keep letters, tokenize, remove stop words and short words

In [ ]:
stop_words = set(stopwords.words("english"))

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]
    return tokens

original_docs = []
clean_docs = []
for d in documents:
    t = preprocess(d)
    if len(t) > 5:                       # drop almost-empty posts
        original_docs.append(d)
        clean_docs.append(" ".join(t))

print("Documents kept:", len(clean_docs))
print(clean_docs[0][:300])

## Document-term matrix (word counts) – LDA works on counts

In [ ]:
vectorizer = CountVectorizer(max_df=0.5, min_df=5, max_features=5000)
# max_df=0.5: ignore words in >50% of docs (too common); min_df=5: ignore words in <5 docs (too rare)
dtm = vectorizer.fit_transform(clean_docs)
words = vectorizer.get_feature_names_out()
print("Document-term matrix:", dtm.shape)

## (b) Apply LDA to find the main topics

In [ ]:
num_topics = 4
lda = LatentDirichletAllocation(n_components=num_topics, random_state=42, learning_method="batch", max_iter=20)
doc_topics = lda.fit_transform(dtm)        # each row: topic mixture of one document (sums to 1)

for k, topic in enumerate(lda.components_):
    top = topic.argsort()[::-1][:10]
    print(f"Topic {k}:", ", ".join(words[i] for i in top))

In [ ]:
# topic mixture of one document
print(original_docs[0][:200], "...")
for k, share in enumerate(doc_topics[0]):
    print(f"  Topic {k}: {share:.2f}")

## (c) Visualise topics and their words

In [ ]:
fig, axes = plt.subplots(1, num_topics, figsize=(18, 5))

for k, topic in enumerate(lda.components_):
    top = topic.argsort()[::-1][:10]
    axes[k].barh([words[i] for i in top][::-1], topic[top][::-1])
    axes[k].set_title(f"Topic {k}")

plt.suptitle("Top 10 words per LDA topic")
plt.tight_layout()
plt.show()

In [ ]:
# how many documents are mostly about each topic
dominant = doc_topics.argmax(axis=1)
counts = np.bincount(dominant, minlength=num_topics)

plt.bar([f"Topic {k}" for k in range(num_topics)], counts)
plt.title("Documents per dominant topic")
plt.ylabel("Documents")
plt.show()